<a href="https://colab.research.google.com/github/Unfavl3mon/Predictive-Mainternance-Vibration/blob/main/Predictive_MA_Vibration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import io, re
import pandas as pd
import numpy as np
from google.colab import files

# ==========================================
# SECTION 1: MACHINERY SELECTION (พร้อมระบบเช็คค่ากรอกผิด)
# ==========================================
print("--- SECTION 1: MACHINERY SETUP ---")

# ตรวจสอบการเลือกกลุ่ม
while True:
    print("\n1: Groups 1 & 3 (Large Machines/Pumps)")
    print("2: Groups 2 & 4 (Medium Machines)")
    g_choice = input("เลือกกลุ่มอุปกรณ์ (1 หรือ 2): ").strip()
    if g_choice in ['1', '2']:
        break
    print("❌ กรุณาเลือกเฉพาะเลข 1 หรือ 2 เท่านั้น")

# ตรวจสอบการเลือกฐาน
while True:
    f_choice = input("เลือกประเภทฐาน Rigid หรือ Flexible (R/F): ").strip().upper()
    if f_choice in ['R', 'F']:
        break
    print("❌ กรอกเฉพาะ R (ฐานแน่น) หรือ F (ฐานยืดหยุ่น)")

# Mapping ข้อมูล
iso_table = {
    "1": {"name": "Groups 1 & 3", "Rigid": [2.3, 4.5, 7.1], "Flexible": [3.5, 7.1, 11.0]},
    "2": {"name": "Groups 2 & 4", "Rigid": [1.4, 2.8, 4.5], "Flexible": [2.3, 3.5, 7.1]}
}

selected_group_data = iso_table[g_choice]
selected_foundation = "Rigid" if f_choice == "R" else "Flexible"
print(f"\n✅ ตั้งค่าเรียบร้อย: {selected_group_data['name']} | Foundation: {selected_foundation}")


# ==========================================
# SECTION 2: FILE UPLOAD & DATA EXTRACTION
# ==========================================
print("\n--- SECTION 2: UPLOAD & DATA EXTRACTION ---")
uploaded = files.upload()

if uploaded:
    f_name = list(uploaded.keys())[0]
    content = uploaded[f_name].decode("utf-8", errors="ignore")

    name_match = re.search(r"Equipment:\s+(.*)", content)
    extracted_name = name_match.group(1).strip() if name_match else "Unknown Device"

    nums = re.findall(r"(?<!-)\b([-+]?\d*\.\d+|\d+)\b", content)
    start_idx = next((i for i, v in enumerate(nums) if float(v) == 0), -1)

    if start_idx != -1:
        v_data = [float(x) for x in nums[start_idx:]]
        amps = v_data[1::2]
        current_rms = np.sqrt(np.mean(np.square(amps)))
        print(f"✅ อ่านไฟล์: {f_name} สำเร็จ")
    else:
        print("❌ ข้อผิดพลาด: ไม่พบโครงสร้างข้อมูลที่ถูกต้องในไฟล์")
        current_rms = None
else:
    print("❌ ข้อผิดพลาด: ไม่มีการเลือกไฟล์")
    current_rms = None


# ==========================================
# SECTION 3: STATUS CHECK (เปรียบเทียบค่า)
# ==========================================
print("\n--- SECTION 3: VIBRATION STATUS CHECK ---")

if current_rms is not None:
    limits = selected_group_data[selected_foundation]

    if current_rms <= limits[0]:
        status_color, desc = "🟢 Zone A", "Good (ปกติ/เครื่องใหม่)"
    elif current_rms <= limits[1]:
        status_color, desc = "🟡 Zone B", "Satisfactory (ใช้งานได้ปกติ)"
    elif current_rms <= limits[2]:
        status_color, desc = "🟠 Zone C", "Warning (เฝ้าระวัง/จำกัดเวลาใช้งาน)"
    else:
        status_color, desc = "🔴 Zone D", "CRITICAL (อันตราย/ควรหยุดเครื่อง)"

    print("=" * 60)
    print(f"DEVICE NAME  : {extracted_name}")
    print(f"ISO STANDARD : {selected_group_data['name']} ({selected_foundation})")
    print(f"MEASURED RMS : {current_rms:.4f} G-s")
    print(f"STATUS       : {status_color} -> {desc}")
    print("=" * 60)
else:
    print("⚠️ ไม่สามารถแสดงผลสรุปได้ เนื่องจากขั้นตอนก่อนหน้าล้มเหลว")

--- SECTION 1: MACHINERY SETUP ---

1: Groups 1 & 3 (Large Machines/Pumps)
2: Groups 2 & 4 (Medium Machines)
เลือกกลุ่มอุปกรณ์ (1 หรือ 2): /
❌ กรุณาเลือกเฉพาะเลข 1 หรือ 2 เท่านั้น

1: Groups 1 & 3 (Large Machines/Pumps)
2: Groups 2 & 4 (Medium Machines)
เลือกกลุ่มอุปกรณ์ (1 หรือ 2): 2
เลือกประเภทฐาน Rigid หรือ Flexible (R/F): F

✅ ตั้งค่าเรียบร้อย: Groups 2 & 4 | Foundation: Flexible

--- SECTION 2: UPLOAD & DATA EXTRACTION ---


Saving A_CH-06 A_NAA_1490__Oct24.txt to A_CH-06 A_NAA_1490__Oct24.txt
✅ อ่านไฟล์: A_CH-06 A_NAA_1490__Oct24.txt สำเร็จ

--- SECTION 3: VIBRATION STATUS CHECK ---
DEVICE NAME  : Motor Compressor OAH-06_A
ISO STANDARD : Groups 2 & 4 (Flexible)
MEASURED RMS : 252.9081 G-s
STATUS       : 🔴 Zone D -> CRITICAL (อันตราย/ควรหยุดเครื่อง)
